# Distribution Shift, Corruption, and Runtime Monitoring Lab


## Purpose

This lab simulates non-adversarial stress: corruption, distribution shift, and monitoring signals.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 2000
d = 12

X = rng.normal(size=(n, d))
true_weights = rng.normal(size=d)
y = ((X @ true_weights) > 0).astype(int)
model_weights = true_weights + rng.normal(scale=0.25, size=d)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def predict(features):
    return (sigmoid(features @ model_weights) >= 0.5).astype(int)

def accuracy(features, labels):
    return float(np.mean(predict(features) == labels))

def adversarial_perturb(features, labels, epsilon):
    direction = np.where(labels[:, None] == 1, -1.0, 1.0)
    return features + epsilon * direction * np.sign(model_weights)

accuracy(X, y)

In [ ]:
rows = []

for shift_strength in np.linspace(0.0, 1.0, 11):
    X_shifted = X + rng.normal(loc=shift_strength, scale=0.15, size=X.shape)
    shifted_accuracy = accuracy(X_shifted, y)

    rows.append({
        "shift_strength": shift_strength,
        "accuracy_under_shift": shifted_accuracy,
        "accuracy_drop": accuracy(X, y) - shifted_accuracy,
    })

shift_results = pd.DataFrame(rows)

shift_results["alert"] = shift_results["accuracy_drop"] > 0.15
shift_results

## Interpretation

Runtime resilience should track performance degradation and trigger review when shift indicators exceed thresholds.